# Transfer Learning - NIH Chest X-Ray Classification

**✨ PyDrive2 Setup - No drive.mount() OAuth prompts!**

This notebook uses PyDrive2 for fast, persistent authentication to Google Drive.

**Runtime**: Python 3 with GPU (T4, A100, or V100 recommended)

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Setup PyDrive2 Authentication

**Using your own Google Cloud OAuth credentials - no "third party" warnings!**

In [ ]:
!pip install -q pydrive2

from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import files
import os

# Step 1: Upload your client_secrets.json
print("📁 Upload your client_secrets.json file")
print("   (Get this from your local project: .colab/client_secrets.json)")
print()

uploaded = files.upload()

# Save to current directory
if 'client_secrets.json' in uploaded:
    with open('client_secrets.json', 'wb') as f:
        f.write(uploaded['client_secrets.json'])
    print("✓ client_secrets.json uploaded")
else:
    raise FileNotFoundError("Please upload client_secrets.json")

# Step 2: Authenticate with your credentials
gauth = GoogleAuth()

# Configure to use the uploaded credentials
gauth.settings['client_config_file'] = 'client_secrets.json'

# Authenticate (opens OAuth prompt on first run)
gauth.LocalWebserverAuth()

# Create Drive client
drive = GoogleDrive(gauth)

print("\n✓ PyDrive2 authenticated with your Google Cloud credentials")
print("✓ No 'third party' warnings!")

## 3. Download Data from Google Drive

**⚠️ IMPORTANT: Setup Instructions**

### Step 1: Prepare Your Google Drive Folder

Upload the **Colab-specific data splits** to Google Drive.

**Files to upload** (from `colab/data_splits/` in the project):
- `train_split.csv` (4.7 MB)
- `val_split.csv` (970 KB)  
- `test_split.csv` (1000 KB)
- `preprocessing_config.json` (400 bytes)

**Why Colab-specific?** These CSV files only contain image filenames (not absolute paths), making them 80% smaller and compatible with any environment.

### Step 2: Get Your Folder ID

From the folder URL:
```
https://drive.google.com/drive/folders/1a2b3c4d5e6f7g8h9
                                        ^^^^^^^^^^^^^^^^
                                        This is your folder ID
```

### Step 3: Update `DATA_FOLDER_ID` in the cell below

The notebook will automatically construct full paths based on where kagglehub downloads the images.

In [ ]:
from pathlib import Path
import time

def download_folder_from_drive(drive, folder_id, destination_dir):
    """Download all files from a Google Drive folder."""
    destination_dir = Path(destination_dir)
    destination_dir.mkdir(parents=True, exist_ok=True)
    
    # List files in folder
    file_list = drive.ListFile({
        'q': f"'{folder_id}' in parents and trashed=false"
    }).GetList()
    
    print(f"📦 Found {len(file_list)} files in Drive folder")
    print(f"📥 Downloading to: {destination_dir}\n")
    
    # Download each file
    for idx, file in enumerate(file_list, 1):
        file_path = destination_dir / file['title']
        print(f"[{idx}/{len(file_list)}] {file['title']:<40} ", end='', flush=True)
        
        start = time.time()
        file.GetContentFile(str(file_path))
        duration = time.time() - start
        
        size_mb = file_path.stat().st_size / 1024 / 1024
        speed_mbps = size_mb / duration if duration > 0 else 0
        print(f"✓ {size_mb:>6.1f} MB ({speed_mbps:.1f} MB/s)")
    
    return len(file_list)

# ⚠️ REPLACE WITH YOUR GOOGLE DRIVE FOLDER ID
DATA_FOLDER_ID = 'YOUR_FOLDER_ID_HERE'  # Get from Drive URL

# Check if already downloaded
destination = Path('/content/data')
required_files = [
    'train_split.csv',
    'val_split.csv', 
    'test_split.csv',
    'preprocessing_config.json'
]

if all((destination / f).exists() for f in required_files):
    print("✓ Data already downloaded")
    csv_files = list(destination.glob('*.csv'))
    json_files = list(destination.glob('*.json'))
    print(f"✓ Found {len(csv_files)} CSV files: {[f.name for f in csv_files]}")
    print(f"✓ Found {len(json_files)} JSON files: {[f.name for f in json_files]}")
else:
    # Download data
    num_files = download_folder_from_drive(drive, DATA_FOLDER_ID, destination)
    print(f"\n✓ Downloaded {num_files} files to {destination}")
    
    # Verify required files
    csv_files = list(destination.glob('*.csv'))
    json_files = list(destination.glob('*.json'))
    print(f"\n📊 CSV files: {[f.name for f in csv_files]}")
    print(f"📋 JSON files: {[f.name for f in json_files]}")
    
    # Check for required files
    missing_files = [f for f in required_files if not (destination / f).exists()]
    
    if missing_files:
        print(f"\n❌ ERROR: Missing required files:")
        for f in missing_files:
            print(f"  - {f}")
        print(f"\n⚠️  Your Google Drive folder must contain these 4 files:")
        for f in required_files:
            print(f"  ✓ {f}")
        print(f"\n💡 These files are created by notebook 03 in the main project.")
        print(f"   Upload them to Google Drive, then update DATA_FOLDER_ID above.")
        raise FileNotFoundError(f"Missing required data files: {missing_files}")
    else:
        print(f"\n✅ All required files found!")

## 4. Imports and Setup

In [ ]:
import json
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from tensorflow.keras.applications import ResNet50, DenseNet121, EfficientNetB3
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs available: {len(gpus)}")
if gpus:
    for gpu in gpus:
        print(f"  - {gpu.name}")

## 5. Configuration and Paths

In [ ]:
# Setup directories
PROJECT_ROOT = Path('/content')
PROCESSED_DIR = Path('/content/data')
MODELS_DIR = PROJECT_ROOT / 'models'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'

MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Data directory: {PROCESSED_DIR}")
print(f"✓ Models directory: {MODELS_DIR}")
print(f"✓ Outputs directory: {OUTPUTS_DIR}")

In [ ]:
# Training configuration
CONFIG = {
    'img_height': 224,
    'img_width': 224,
    'channels': 3,
    'batch_size': 64,  # Increase to 128 for A100
    'epochs_stage1': 5,
    'epochs_stage2': 10,
    'learning_rate_stage1': 0.001,
    'learning_rate_stage2': 0.0001,
    'unfreeze_layers': 20,
    'dense_units': 512,
    'dropout_rate': 0.5,
    'early_stopping_patience': 10,
    'reduce_lr_patience': 5,
    'num_classes': 14,
    'use_sample': True,  # Set to False for full training
    'sample_size': 1000,
    'random_state': 42
}

# Models to train
MODELS_TO_TRAIN = ['resnet50', 'densenet121', 'efficientnetb3']

print("⚙️  Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")
print(f"\n🎯 Models to train: {MODELS_TO_TRAIN}")

## 6. Download NIH Chest X-Ray Images

**⚠️ This downloads ~1 GB of sample images for testing**

For full training, you'll need to download all ~47 GB of images from Kaggle.

In [ ]:
# Check if dataset already cached (Colab Pro persistent disk)
CACHE_DIR = Path('/content/data/nih-chest-xrays')

if CACHE_DIR.exists() and any(CACHE_DIR.glob('images_*')):
    print("✅ Found cached NIH Chest X-Ray dataset!")
    print(f"📂 Location: {CACHE_DIR}")
    IMAGE_DIR = CACHE_DIR
    
    # Verify cache
    image_subdirs = list(CACHE_DIR.glob('images_*'))
    print(f"📊 Image directories: {len(image_subdirs)}")
    for subdir in sorted(image_subdirs)[:3]:
        num_images = len(list((subdir / 'images').glob('*.png')))
        print(f"  {subdir.name}/: {num_images:,} images")
    
    print("\n💡 Using cached data - no download needed!")
    
else:
    # Download from Kaggle (first time or cache not found)
    print("📥 Downloading NIH Chest X-Ray dataset from Kaggle...")
    print("⚠️  This may take 10-30 minutes depending on your connection")
    print("💡 Dataset size: ~47 GB (112,120 images)")
    print("\n🔄 With Colab Pro persistent disk, this is a ONE-TIME download")
    print("   Future sessions will use the cached data.\n")
    
    import kagglehub
    
    dataset_path = kagglehub.dataset_download("nih-chest-xrays/data")
    print(f"\n✓ Dataset downloaded to: {dataset_path}")
    
    # Move to persistent location if needed
    dataset_path = Path(dataset_path)
    if dataset_path != CACHE_DIR:
        print(f"\n📦 Copying to persistent disk: {CACHE_DIR}")
        import shutil
        CACHE_DIR.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(dataset_path, CACHE_DIR)
        print("✓ Copied to persistent disk")
    
    IMAGE_DIR = CACHE_DIR

# Show final location
print(f"\n📂 Using images from: {IMAGE_DIR}")
print(f"💾 Storage type: {'Persistent disk (Pro)' if IMAGE_DIR == CACHE_DIR else 'Ephemeral'}")

## 7. Load and Update Data Splits

**The CSV files contain local paths - we need to update them to point to Colab's images**

In [ ]:
# Load CSV files
print("📥 Loading data splits...")
train_df = pd.read_csv(PROCESSED_DIR / 'train_split.csv')
val_df = pd.read_csv(PROCESSED_DIR / 'val_split.csv')
test_df = pd.read_csv(PROCESSED_DIR / 'test_split.csv')

print(f"✓ Train: {len(train_df):,} images")
print(f"✓ Val:   {len(val_df):,} images")
print(f"✓ Test:  {len(test_df):,} images")

# Load preprocessing config
with open(PROCESSED_DIR / 'preprocessing_config.json', 'r') as f:
    prep_config = json.load(f)

disease_classes = prep_config['disease_classes']

print(f"\n🏥 Disease classes ({len(disease_classes)}):")
for i, disease in enumerate(disease_classes, 1):
    print(f"  {i:2d}. {disease}")

# Build image paths from Image Index
print(f"\n🔄 Building image paths...")

def build_image_path(filename, base_dir):
    """
    Find image file in the downloaded Kaggle dataset.
    
    NIH dataset has images in subdirectories: images_001/images/, images_002/images/, etc.
    """
    # Search all image subdirectories
    for subdir in sorted(Path(base_dir).glob('images_*')):
        if subdir.is_dir():
            img_path = subdir / 'images' / filename
            if img_path.exists():
                return str(img_path)
    
    # If not found, return None (will be caught in validation)
    return None

# Build full_path column for all splits
print("  Building train paths...", end=' ', flush=True)
train_df['full_path'] = train_df['Image Index'].apply(lambda x: build_image_path(x, IMAGE_DIR))
print("✓")

print("  Building val paths...", end=' ', flush=True)
val_df['full_path'] = val_df['Image Index'].apply(lambda x: build_image_path(x, IMAGE_DIR))
print("✓")

print("  Building test paths...", end=' ', flush=True)
test_df['full_path'] = test_df['Image Index'].apply(lambda x: build_image_path(x, IMAGE_DIR))
print("✓")

# Verify that images exist
print(f"\n🔍 Verifying images...")
all_dfs = [('train', train_df), ('val', val_df), ('test', test_df)]
total_missing = 0

for split_name, df in all_dfs:
    missing_in_split = df['full_path'].isna().sum()
    total_missing += missing_in_split
    
    if missing_in_split > 0:
        print(f"  ❌ {split_name}: {missing_in_split:,} missing images")
        # Show a few examples
        missing_files = df[df['full_path'].isna()]['Image Index'].head(3).tolist()
        for fname in missing_files:
            print(f"     - {fname}")
    else:
        print(f"  ✓ {split_name}: All {len(df):,} images found")

if total_missing > 0:
    print(f"\n⚠️  Total {total_missing:,} images not found!")
    print("   This might indicate:")
    print("   - Incomplete kagglehub download")
    print("   - CSV files from different dataset version")
    raise FileNotFoundError(f"{total_missing} image files not found in {IMAGE_DIR}")
else:
    print(f"\n✅ All {len(train_df) + len(val_df) + len(test_df):,} images verified!")

In [ ]:
# Load CSV files
train_df = pd.read_csv(PROCESSED_DIR / 'train_split.csv')
val_df = pd.read_csv(PROCESSED_DIR / 'val_split.csv')
test_df = pd.read_csv(PROCESSED_DIR / 'test_split.csv')

# Load preprocessing config
with open(PROCESSED_DIR / 'preprocessing_config.json', 'r') as f:
    prep_config = json.load(f)

disease_classes = prep_config['disease_classes']

print(f"✓ Train: {len(train_df):,} images")
print(f"✓ Val:   {len(val_df):,} images")
print(f"✓ Test:  {len(test_df):,} images")
print(f"\n🏥 Disease classes ({len(disease_classes)}):")
for i, disease in enumerate(disease_classes, 1):
    print(f"  {i:2d}. {disease}")

In [ ]:
# Sample data if requested
if CONFIG['use_sample']:
    sample_size = CONFIG['sample_size']
    train_df = train_df.sample(n=min(sample_size, len(train_df)), random_state=42)
    val_df = val_df.sample(n=min(sample_size // 5, len(val_df)), random_state=42)
    test_df = test_df.sample(n=min(sample_size // 5, len(test_df)), random_state=42)
    
    print(f"⚠️  Using SAMPLE mode (for testing):")
    print(f"  Train: {len(train_df):,} images")
    print(f"  Val:   {len(val_df):,} images")
    print(f"  Test:  {len(test_df):,} images")
    print(f"\n💡 Set CONFIG['use_sample'] = False for full training")

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

## 7. Data Generators

In [ ]:
# Training data augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)

# Validation data (no augmentation)
val_datagen = ImageDataGenerator(rescale=1./255)

# Create generators
train_gen = train_datagen.flow_from_dataframe(
    train_df,
    x_col='full_path',
    y_col=disease_classes,
    target_size=(CONFIG['img_height'], CONFIG['img_width']),
    batch_size=CONFIG['batch_size'],
    class_mode='raw',
    shuffle=True
)

val_gen = val_datagen.flow_from_dataframe(
    val_df,
    x_col='full_path',
    y_col=disease_classes,
    target_size=(CONFIG['img_height'], CONFIG['img_width']),
    batch_size=CONFIG['batch_size'],
    class_mode='raw',
    shuffle=False
)

print(f"✓ Train batches: {len(train_gen)}")
print(f"✓ Val batches: {len(val_gen)}")
print(f"✓ Steps per epoch: ~{len(train_gen)}")

## 8. Model Building Functions

In [ ]:
def build_transfer_model(base_model_class, model_name, config):
    """
    Build a transfer learning model.
    
    Args:
        base_model_class: keras.applications model class
        model_name: Name for the model
        config: Configuration dictionary
    
    Returns:
        model: Compiled keras Model
        base_model: Base model (for unfreezing layers)
    """
    input_shape = (config['img_height'], config['img_width'], config['channels'])
    
    # Load pre-trained base model
    base_model = base_model_class(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    
    # Freeze base model initially
    base_model.trainable = False
    
    # Build top layers
    inputs = keras.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(config['dense_units'], activation='relu')(x)
    x = layers.Dropout(config['dropout_rate'])(x)
    outputs = layers.Dense(config['num_classes'], activation='sigmoid')(x)
    
    model = keras.Model(inputs, outputs, name=model_name)
    
    return model, base_model

## 9. Training Function

In [ ]:
def train_model(model_class, model_name, train_gen, val_gen, config):
    """
    Train a transfer learning model in two stages:
    1. Feature extraction (frozen base)
    2. Fine-tuning (unfrozen top layers)
    """
    # Skip if not in training list
    if model_name.lower() not in [m.lower() for m in MODELS_TO_TRAIN]:
        print(f"\n⏭️  Skipping {model_name} (not in MODELS_TO_TRAIN)")
        return None, None
    
    print(f"\n{'='*60}")
    print(f"🚀 TRAINING: {model_name}")
    print(f"{'='*60}")
    
    # Build model
    model, base_model = build_transfer_model(
        model_class, 
        f"{model_name.lower()}_transfer", 
        config
    )
    
    print(f"\n📐 Architecture:")
    print(f"  Base model: {model_name}")
    print(f"  Base layers: {len(base_model.layers)} (frozen)")
    print(f"  Total parameters: {model.count_params():,}")
    trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
    print(f"  Trainable parameters: {trainable_params:,}")
    
    # ========================================
    # STAGE 1: Feature Extraction
    # ========================================
    print(f"\n{'-'*60}")
    print("🔹 STAGE 1: Feature Extraction (frozen base)")
    print(f"{'-'*60}")
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config['learning_rate_stage1']),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc', multi_label=True)]
    )
    
    history_s1 = model.fit(
        train_gen,
        epochs=config['epochs_stage1'],
        validation_data=val_gen,
        callbacks=[
            callbacks.EarlyStopping(
                monitor='val_auc',
                mode='max',
                patience=config['early_stopping_patience'] // 2,
                restore_best_weights=True,
                verbose=1
            ),
            callbacks.ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=config['reduce_lr_patience'] // 2,
                verbose=1
            )
        ],
        verbose=1
    )
    
    # ========================================
    # STAGE 2: Fine-Tuning
    # ========================================
    print(f"\n{'-'*60}")
    print("🔸 STAGE 2: Fine-Tuning (unfrozen top layers)")
    print(f"{'-'*60}")
    
    # Unfreeze top layers
    base_model.trainable = True
    for layer in base_model.layers[:-config['unfreeze_layers']]:
        layer.trainable = False
    
    trainable_layers = sum([1 for layer in base_model.layers if layer.trainable])
    print(f"  Unfrozen layers: {trainable_layers} / {len(base_model.layers)}")
    
    # Recompile with lower learning rate
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config['learning_rate_stage2']),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc', multi_label=True)]
    )
    
    model_save_path = MODELS_DIR / f"{model_name.lower()}_transfer_best.keras"
    
    history_s2 = model.fit(
        train_gen,
        epochs=config['epochs_stage2'],
        validation_data=val_gen,
        callbacks=[
            callbacks.ModelCheckpoint(
                str(model_save_path),
                monitor='val_auc',
                mode='max',
                save_best_only=True,
                verbose=1
            ),
            callbacks.EarlyStopping(
                monitor='val_auc',
                mode='max',
                patience=config['early_stopping_patience'],
                restore_best_weights=True,
                verbose=1
            ),
            callbacks.ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=config['reduce_lr_patience'],
                verbose=1
            )
        ],
        verbose=1
    )
    
    print(f"\n✓ Model saved: {model_save_path}")
    
    return model, {'stage1': history_s1.history, 'stage2': history_s2.history}

## 10. Train All Models

**This will train 3 models sequentially. Estimated time:**
- Sample mode (1000 images): ~15 minutes per model
- Full dataset (78K images): ~2-3 hours per model

In [ ]:
import time

trained_models = {}
training_histories = {}

start_time = time.time()

for model_name, model_class in [
    ('ResNet50', ResNet50),
    ('DenseNet121', DenseNet121),
    ('EfficientNetB3', EfficientNetB3)
]:
    model_start = time.time()
    
    model, history = train_model(
        model_class=model_class,
        model_name=model_name,
        train_gen=train_gen,
        val_gen=val_gen,
        config=CONFIG
    )
    
    if model is not None:
        trained_models[model_name] = model
        training_histories[model_name] = history
        
        model_duration = time.time() - model_start
        print(f"\n⏱️  {model_name} training time: {model_duration/60:.1f} minutes")

total_duration = time.time() - start_time

print(f"\n{'='*60}")
print("🎉 TRAINING COMPLETE")
print(f"{'='*60}")
print(f"\n✓ Trained {len(trained_models)} models:")
for name in trained_models.keys():
    print(f"  • {name}")
print(f"\n⏱️  Total training time: {total_duration/60:.1f} minutes ({total_duration/3600:.2f} hours)")

## 11. Visualize Training History

In [ ]:
for model_name, history in training_histories.items():
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f'{model_name} Training History', fontsize=16, fontweight='bold')
    
    s1_epochs = len(history['stage1']['loss'])
    s2_epochs = len(history['stage2']['loss'])
    total_epochs = s1_epochs + s2_epochs
    
    epochs_s1 = list(range(1, s1_epochs + 1))
    epochs_s2 = list(range(s1_epochs + 1, total_epochs + 1))
    
    for idx, metric in enumerate(['loss', 'accuracy', 'auc']):
        ax = axes[idx]
        
        train_s1 = history['stage1'][metric]
        val_s1 = history['stage1'][f'val_{metric}']
        train_s2 = history['stage2'][metric]
        val_s2 = history['stage2'][f'val_{metric}']
        
        # Plot Stage 1
        ax.plot(epochs_s1, train_s1, 'b-', label='Train S1 (frozen)', alpha=0.7, linewidth=2)
        ax.plot(epochs_s1, val_s1, 'b--', label='Val S1 (frozen)', alpha=0.7, linewidth=2)
        
        # Plot Stage 2
        ax.plot(epochs_s2, train_s2, 'r-', label='Train S2 (fine-tune)', alpha=0.7, linewidth=2)
        ax.plot(epochs_s2, val_s2, 'r--', label='Val S2 (fine-tune)', alpha=0.7, linewidth=2)
        
        # Mark transition
        ax.axvline(x=s1_epochs, color='gray', linestyle=':', alpha=0.5, linewidth=2)
        ax.text(s1_epochs, ax.get_ylim()[1]*0.95, 'Unfreeze', 
                ha='center', va='top', fontsize=9, color='gray')
        
        ax.set_xlabel('Epoch', fontsize=11)
        ax.set_ylabel(metric.capitalize(), fontsize=11)
        ax.set_title(metric.capitalize(), fontsize=12, fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # Save figure
    fig_path = OUTPUTS_DIR / f'{model_name.lower()}_training_history.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"✓ Saved: {fig_path}")
    
    plt.show()

## 12. Evaluate Models on Test Set

In [ ]:
test_datagen = ImageDataGenerator(rescale=1./255)

test_gen = test_datagen.flow_from_dataframe(
    test_df,
    x_col='full_path',
    y_col=disease_classes,
    target_size=(CONFIG['img_height'], CONFIG['img_width']),
    batch_size=CONFIG['batch_size'],
    class_mode='raw',
    shuffle=False
)

results = {}

print("📊 Evaluating models on test set...\n")

for model_name, model in trained_models.items():
    print(f"Evaluating {model_name}...", end=' ', flush=True)
    
    test_loss, test_acc, test_auc = model.evaluate(test_gen, verbose=0)
    
    results[model_name] = {
        'loss': test_loss,
        'accuracy': test_acc,
        'auc': test_auc
    }
    
    print(f"✓ Loss: {test_loss:.4f} | Acc: {test_acc:.4f} | AUC: {test_auc:.4f}")

print(f"\n{'='*60}")
print("✅ EVALUATION COMPLETE")
print(f"{'='*60}")

## 13. Compare Model Performance

In [ ]:
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('auc', ascending=False)

print("\n🏆 Model Comparison (sorted by AUC):")
print("="*60)
print(results_df.to_string())
print("="*60)

# Visualize comparison
fig, ax = plt.subplots(figsize=(10, 6))
results_df[['accuracy', 'auc']].plot(kind='bar', ax=ax, width=0.8)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_ylim(0, 1)
ax.legend(['Accuracy', 'AUC'], fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

# Save figure
fig_path = OUTPUTS_DIR / 'model_comparison.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f"\n✓ Saved: {fig_path}")

plt.show()

# Find best model
best_model = results_df.index[0]
best_auc = results_df.loc[best_model, 'auc']
print(f"\n🥇 Best Model: {best_model} (AUC: {best_auc:.4f})")

## 14. Upload Results Back to Google Drive

**This uploads trained models and outputs back to your Drive folder**

In [ ]:
def upload_file_to_drive(drive, local_path, drive_folder_id):
    """Upload a file to Google Drive."""
    local_path = Path(local_path)
    
    file_metadata = {
        'title': local_path.name,
        'parents': [{'id': drive_folder_id}]
    }
    
    file = drive.CreateFile(file_metadata)
    file.SetContentFile(str(local_path))
    
    print(f"📤 Uploading {local_path.name}... ", end='', flush=True)
    file.Upload()
    size_mb = local_path.stat().st_size / 1024 / 1024
    print(f"✓ ({size_mb:.1f} MB)")
    print(f"   URL: https://drive.google.com/file/d/{file['id']}")
    
    return file['id']

# Upload models
print("\n📦 Uploading models to Google Drive...\n")
RESULTS_FOLDER_ID = DATA_FOLDER_ID  # Or create separate results folder

for model_file in MODELS_DIR.glob('*.keras'):
    upload_file_to_drive(drive, model_file, RESULTS_FOLDER_ID)

# Upload figures
print("\n📊 Uploading figures...\n")
for fig_file in OUTPUTS_DIR.glob('*.png'):
    upload_file_to_drive(drive, fig_file, RESULTS_FOLDER_ID)

print("\n✅ All results uploaded to Drive!")

## 15. Training Summary

In [ ]:
print("\n" + "="*60)
print("📋 TRAINING SUMMARY")
print("="*60)

print(f"\n🎯 Configuration:")
print(f"  Dataset: NIH Chest X-Ray (14 disease classes)")
print(f"  Train images: {len(train_df):,}")
print(f"  Val images: {len(val_df):,}")
print(f"  Test images: {len(test_df):,}")
print(f"  Batch size: {CONFIG['batch_size']}")
print(f"  Image size: {CONFIG['img_height']}x{CONFIG['img_width']}")

print(f"\n🏆 Model Performance:")
for model_name, metrics in results.items():
    print(f"  {model_name}:")
    print(f"    Accuracy: {metrics['accuracy']:.4f}")
    print(f"    AUC: {metrics['auc']:.4f}")
    print(f"    Loss: {metrics['loss']:.4f}")

print(f"\n⏱️  Training Time: {total_duration/60:.1f} minutes ({total_duration/3600:.2f} hours)")

print(f"\n💾 Saved Files:")
print(f"  Models: {list(MODELS_DIR.glob('*.keras'))}")
print(f"  Figures: {list(OUTPUTS_DIR.glob('*.png'))}")

print(f"\n✅ Training complete! All results saved to Google Drive.")
print("="*60)